In [13]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

In [14]:
universe = {
    "Maybank": {
        "ticker": "1155.KL",
        "aliases": [
            "Maybank",
            "Malayan Banking"
        ]
    },

    "CIMB": {
        "ticker": "1023.KL",
        "aliases": [
            "CIMB",
            "CIMB Group"
        ]
    },

    "Public Bank": {
        "ticker": "1295.KL",
        "aliases": [
            "Public Bank"
        ]
    },

    "Tenaga Nasional": {
        "ticker": "5347.KL",
        "aliases": [
            "Tenaga Nasional",
            "TNB"
        ]
    },

    "Petronas Chemicals": {
        "ticker": "5183.KL",
        "aliases": [
            "Petronas Chemicals",
            "PETRONAS Chemicals",
            "PCG"
        ]
    }
}

In [ ]:
end_date = pd.to_datetime("2023-01-01")
end_date = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
start_date = end_date - pd.DateOffset(years=5)
print("Start:", start_date)
print("End:", end_date)

Start: 2021-11-10 00:00:00
End: 2026-11-10 00:00:00


In [ ]:
def download_market_data(universe,start_date,end_date):
    all_data = []
    for company, info in universe.items():
        ticker = info["ticker"]
        print(f"Downloading {company} ({ticker})...")
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=True,
            progress=False
        )
        if df.empty:
            print(f"No data for {ticker}")
            continue
        if isinstance(df.columns,pd.MultiIndex):
            df.columns = (df.columns.get_level_values(0))
        df = df.reset_index()
        df.columns = [col.lower() for col in df.columns]
        df["ticker"] = ticker
        df["company"] = company
        all_data.append(df)

    market_data = pd.concat(all_data,ignore_index=True)
    return market_data

In [ ]:
data = download_market_data(universe,start_date,end_date)
display(data.head())
print(data.shape)

,date,close,high,low,open,volume,ticker,company
0,2021-11-10,5.846909,5.854227,5.839591,5.854227,3811100,1155.KL,Maybank
1,2021-11-11,5.854227,5.876180,5.846909,5.854227,9120000,1155.KL,Maybank
2,2021-11-12,5.934722,5.949358,5.883498,5.883498,21463000,1155.KL,Maybank
3,2021-11-15,5.927405,6.000583,5.898134,5.934723,5426000,1155.KL,Maybank
4,2021-11-16,5.920086,5.949357,5.898133,5.942040,9802500,1155.KL,Maybank


(6115, 8)


In [ ]:
print(data.info())
print(data.isnull().sum())
print(data["company"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6115 entries, 0 to 6114
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     6115 non-null   datetime64[ns]
 1   close    6115 non-null   float64       
 2   high     6115 non-null   float64       
 3   low      6115 non-null   float64       
 4   open     6115 non-null   float64       
 5   volume   6115 non-null   int64         
 6   ticker   6115 non-null   object        
 7   company  6115 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 382.3+ KB
None
date       0
close      0
high       0
low        0
open       0
volume     0
ticker     0
company    0
dtype: int64
company
Maybank               1223
CIMB                  1223
Public Bank           1223
Tenaga Nasional       1223
Petronas Chemicals    1223
Name: count, dtype: int64


In [ ]:
data["daily_return"] = (data.groupby("ticker")["close"].pct_change(fill_method=None)* 100)
data["intraday_return"] = ((data["close"]-data["open"])/data["open"]* 100)
data["daily_range"] = ((data["high"]-data["low"])/data["open"]* 100)
data["volume_change"] = (data.groupby("ticker")["volume"].pct_change(fill_method=None)* 100)
data["direction"] = np.select([data["daily_return"] > 0, data["daily_return"] < 0],[1, -1],default=0)

In [ ]:
data = data.drop_duplicates(subset=["date","ticker"])
data = data.sort_values(["ticker","date"]).reset_index(drop=True)

In [ ]:
def add_technical_indicators(stock_df):
    stock_df = (stock_df.sort_values("date").copy())
    close = stock_df["close"]
    high = stock_df["high"]
    low = stock_df["low"]
    # ========================================================
    # SMA
    # ========================================================
    stock_df["sma_20"] = (close.rolling(20).mean())
    stock_df["sma_50"] = (close.rolling(50).mean())
    stock_df["sma_200"] = (close.rolling(200).mean())
    # ========================================================
    # EMA
    # ========================================================
    stock_df["ema_12"] = (close.ewm(span=12,adjust=False).mean())
    stock_df["ema_26"] = (close.ewm(span=26,adjust=False).mean())
    # ========================================================
    # MACD
    # ========================================================
    stock_df["macd"] = (stock_df["ema_12"]-stock_df["ema_26"])
    stock_df["macd_signal"] = (stock_df["macd"].ewm(span=9,adjust=False).mean())
    stock_df["macd_histogram"] = (stock_df["macd"]-stock_df["macd_signal"])
    # ========================================================
    # RSI 14
    # ========================================================
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta.clip(upper=0))
    avg_gain = gain.ewm(alpha=1 / 14,adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / 14,adjust=False).mean()
    rs = (avg_gain/avg_loss)
    stock_df["rsi_14"] = (100-(100/(1 + rs)))
    # ========================================================
    # ATR 14
    # ========================================================
    previous_close = close.shift(1)
    true_range = pd.concat([high - low,(high - previous_close).abs(),(low - previous_close).abs()],axis=1).max(axis=1)
    stock_df["atr_14"] = (true_range.ewm(alpha=1 / 14,adjust=False).mean())
    # ========================================================
    # BOLLINGER BANDS
    # ========================================================
    rolling_mean = (close.rolling(20).mean())
    rolling_std = (close.rolling(20).std())
    stock_df["bollinger_middle"] = (rolling_mean)
    stock_df["bollinger_upper"] = (rolling_mean + 2 * rolling_std)
    stock_df["bollinger_lower"] = (rolling_mean-2 * rolling_std)
    stock_df["bollinger_width"] = ((stock_df["bollinger_upper"]-stock_df["bollinger_lower"])/stock_df["bollinger_middle"]* 100)
    # ========================================================
    # VOLATILITY
    # ========================================================
    stock_df["volatility_20"] = (stock_df["daily_return"].rolling(20).std()*np.sqrt(252))
    # ========================================================
    # MOMENTUM
    # ========================================================
    stock_df["momentum_10"] = (close.pct_change(periods=10,fill_method=None)* 100)
    return stock_df

In [ ]:
feature_frames = []
for ticker, stock_df in data.groupby("ticker"):
    stock_features = (add_technical_indicators(stock_df))
    feature_frames.append(stock_features)
market_features = pd.concat(feature_frames,ignore_index=True)
market_features = market_features.sort_values(["ticker","date"]).reset_index(drop=True)

In [ ]:
display(market_features.tail())

,date,close,high,low,open,volume,ticker,company,daily_return,intraday_return,...,macd_signal,macd_histogram,rsi_14,atr_14,bollinger_middle,bollinger_upper,bollinger_lower,bollinger_width,volatility_20,momentum_10
6110,2026-11-02,13.30,13.30,13.16,13.30,2762500,5347.KL,Tenaga Nasional,0.000000,0.000000,...,-0.004955,0.046455,58.062449,0.185596,13.092,13.425252,12.758748,5.090930,10.356756,2.465336
6111,2026-11-03,13.16,13.32,13.14,13.30,3139400,5347.KL,Tenaga Nasional,-1.052634,-1.052634,...,0.002992,0.031790,49.668318,0.185196,13.105,13.426804,12.783195,4.911170,10.720555,0.765692
6112,2026-11-04,13.14,13.22,13.06,13.22,2619900,5347.KL,Tenaga Nasional,-0.151972,-0.605143,...,0.007900,0.019629,48.587677,0.183397,13.119,13.419589,12.818411,4.582502,10.654442,1.232672
6113,2026-11-05,13.10,13.16,13.02,13.16,3353100,5347.KL,Tenaga Nasional,-0.304414,-0.455923,...,0.009988,0.008352,46.412657,0.180297,13.129,13.411687,12.846313,4.306293,10.724073,-1.057397
6114,2026-11-06,13.04,13.04,12.90,13.00,3884300,5347.KL,Tenaga Nasional,-0.458018,0.307692,...,0.009219,-0.003074,43.282750,0.181704,13.137,13.398260,12.875740,3.977474,10.864817,-1.954889


In [24]:
print(market_features.columns.tolist())

['date', 'close', 'high', 'low', 'open', 'volume', 'ticker', 'company', 'daily_return', 'intraday_return', 'daily_range', 'volume_change', 'direction', 'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_histogram', 'rsi_14', 'atr_14', 'bollinger_middle', 'bollinger_upper', 'bollinger_lower', 'bollinger_width', 'volatility_20', 'momentum_10']


In [ ]:
latest_market_data = (market_features.sort_values("date").groupby("ticker",as_index=False).tail(1))
display(
    latest_market_data[
        [
            "date",
            "company",
            "ticker",
            "close",
            "daily_return",
            "rsi_14",
            "macd",
            "macd_signal",
            "volatility_20",
            "momentum_10"
        ]
    ])

,date,company,ticker,close,daily_return,rsi_14,macd,macd_signal,volatility_20,momentum_10
1222,2026-11-06,CIMB,1023.KL,7.06,-1.258743,41.976867,-0.144608,-0.158874,25.147971,-1.258743
4891,2026-11-06,Petronas Chemicals,5183.KL,4.20,-2.552207,40.100394,0.015892,0.041832,46.236751,-9.677425
2445,2026-11-06,Maybank,1155.KL,10.12,-0.393700,55.607534,0.003634,-0.018504,14.343221,0.596416
3668,2026-11-06,Public Bank,1295.KL,4.90,-1.209676,60.280551,0.027282,0.002263,21.365777,3.157897
6114,2026-11-06,Tenaga Nasional,5347.KL,13.04,-0.458018,43.282750,0.006146,0.009219,10.864817,-1.954889


In [ ]:
output_dir = Path("../data/preprocessed_data")
output_dir.mkdir(parents=True,exist_ok=True)
market_features.to_csv(output_dir/ "malaysia_market_features.csv",index=False)
print("Market features saved.")

Market features saved.


In [27]:
check_cols = [
    "date",
    "company",
    "ticker",
    "close",
    "daily_return",
    "rsi_14",
    "macd",
    "macd_signal",
    "atr_14",
    "volatility_20",
    "momentum_10"
]
latest = (market_features.sort_values("date").groupby("ticker").tail(1))
display(latest[check_cols])

,date,company,ticker,close,daily_return,rsi_14,macd,macd_signal,atr_14,volatility_20,momentum_10
1222,2026-11-06,CIMB,1023.KL,7.06,-1.258743,41.976867,-0.144608,-0.158874,0.145449,25.147971,-1.258743
4891,2026-11-06,Petronas Chemicals,5183.KL,4.20,-2.552207,40.100394,0.015892,0.041832,0.189108,46.236751,-9.677425
2445,2026-11-06,Maybank,1155.KL,10.12,-0.393700,55.607534,0.003634,-0.018504,0.121156,14.343221,0.596416
3668,2026-11-06,Public Bank,1295.KL,4.90,-1.209676,60.280551,0.027282,0.002263,0.095025,21.365777,3.157897
6114,2026-11-06,Tenaga Nasional,5347.KL,13.04,-0.458018,43.282750,0.006146,0.009219,0.181704,10.864817,-1.954889
